Import Necessary Libraries

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import re
from tqdm import tqdm
import requests

GITHUB Details - Replace GITHUB ACCESS KEY with your own Personal access token

In [ ]:
github_token = "GITHUB ACCESS KEY"
headers = {"Authorization": f"token {github_token}"}

Discussion to Issues

In [ ]:
df = pd.read_csv("../Dataset/DiscussionToIssue.csv")

In [ ]:
encoder = LabelEncoder()
df['IsIssueRaised'] = encoder.fit_transform(df['IsIssueRaised'])
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df_shuffled['concatenated'] = df_shuffled['Title'] + ' ' + df_shuffled['Description'] + ' ' + df_shuffled['Comments']

Training ...

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_shuffled['concatenated'], df['IsIssueRaised'], test_size=0.2, random_state=42
)

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_vec, y_train)

Testing ...

In [ ]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Discussion to Issues with Description alone

In [ ]:
df_shuffled['concatenated'] = df_shuffled['Title'] + ' ' + df_shuffled['Description']

Training ...

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_shuffled['concatenated'], df['IsIssueRaised'], test_size=0.2, random_state=42
)

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_vec, y_train)

Testing ...

In [ ]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Discussion to Issues with First Comment Alone

Function to extract Repository and Discussion number

In [ ]:
def extract_github_path(url):
    match = re.search(r'github\.com/([^?#]*)', url)
    return match.group(1) if match else None

Download the First Comment of the Discussion

In [ ]:
df_shuffled['Comment'] =  None
for index,row in tqdm(df_shuffled.iterrows()):
  repo = extract_github_path(row['Issue'])
  curl = f'https://api.github.com/repos/{repo}/comments'
  repo_comment=[]
  cresponse = requests.get(curl,  headers=headers)
  if cresponse.status_code == 200:
    issueComments = cresponse.json()
    cnt=0
    for comment in issueComments:
        repo_comment.append(comment['body'])
        cnt+=1
        if(cnt<1):
          break
  df_shuffled.at[index, 'Comment'] = repo_comment

In [ ]:
df_shuffled['concatenated'] = df_shuffled['Title'] + ' ' + df_shuffled['Description']+' '+str(df_shuffled['Comment'])

Training ...

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_vec, y_train)

Testing ...

In [ ]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Issues to Discussion

In [ ]:
df1 = pd.read_csv("../Dataset/IssueToDiscussion.csv")

In [ ]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df1_shuffled['concatenated'] = df1_shuffled['Title'] + ' ' + df1_shuffled['Description'] + ' ' + df1_shuffled['Comments']

Training ...

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df1_shuffled['concatenated'], df1['ConvertedFromIssue'], test_size=0.2, random_state=42
)

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_vec, y_train)

Testing ...

In [ ]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Issues to Discussion with Description alone

In [ ]:
df1 = pd.read_csv("../Dataset/IssueToDiscussion.csv")

In [ ]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df1_shuffled['concatenated'] = df1_shuffled['Description']

Training ...

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df1_shuffled['concatenated'], df1['ConvertedFromIssue'], test_size=0.2, random_state=42
)

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_vec, y_train)

Testing ...

In [ ]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Description+ First Comment

In [ ]:
df1 = pd.read_csv("../Dataset/IssueToDiscussion.csv")

Function to extract Repository and Issue Number

In [ ]:
def extract_github_path(url):
    match = re.search(r'github\.com/([^?#]*)', url)
    return match.group(1) if match else None

Download the First Comment

In [ ]:
df1['Comment'] =  None
for index,row in tqdm(df1.iterrows()):
  repo = extract_github_path(row['Issue'])
  curl = f'https://api.github.com/repos/{repo}/comments'
  repo_comment=[]
  cresponse = requests.get(curl,  headers=headers)
  if cresponse.status_code == 200:
    issueComments = cresponse.json()
    cnt=0
    for comment in issueComments:
        repo_comment.append(comment['body'])
        cnt+=1
        if(cnt<1):
          break
  df1.at[index, 'Comment'] = repo_comment

In [ ]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df1_shuffled['concatenated'] = df1_shuffled['Description'] + ' ' + str(df1_shuffled['Comment'])

Training ...

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df1_shuffled['concatenated'], df1['ConvertedFromIssue'], test_size=0.2, random_state=42
)

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_vec, y_train)

Testing ...

In [ ]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Issues to Discussion with Comments

In [ ]:
df1 = pd.read_csv("../Dataset/IssueToDiscussion.csv")

In [ ]:
encoder = LabelEncoder()
df1['ConvertedFromIssue'] = encoder.fit_transform(df1['ConvertedFromIssue'])
df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df1_shuffled['concatenated'] = df1_shuffled['Comments']

Training ...

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df1_shuffled['concatenated'], df1['ConvertedFromIssue'], test_size=0.2, random_state=42
)

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_vec, y_train)

Testing ...

In [ ]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))